# Assignment 1 — EDA ve Baseline Modeller
**XACLE Challenge 2027** (IEEE ICASSP 2027 – The 2nd X-to-Audio Alignment Challenge)

Görev: Bir ses kaydı ile metin açıklaması arasındaki anlamsal uyum için insanların verdiği puanı tahmin etmek.

İçerik:
1. Kurulum
2. Veri kümesine genel bakış
3. Puan dağılımı
4. Dinleyiciler arası tutarlılık
5. Ses üretim modellerine göre analiz
6. Metin analizi
7. Ses analizi
8. Baseline modeller

## 1. Kurulum

In [1]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
REPO_URL = "https://github.com/deafiduck/.-XACLE-IEEE-ICASSP-2027---The-2nd-X-to-Audio-Alignment-Challenge-.git"

if IN_COLAB:
    # Kernel Colab'da çalışıyor: src/ kodunu GitHub'dan çek (önce push etmiş olun!)
    REPO_DIR = Path("/content/repo")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    REPO_DIR = Path.cwd()

sys.path.insert(0, str(REPO_DIR))
print("IN_COLAB:", IN_COLAB, "| REPO_DIR:", REPO_DIR)

IN_COLAB: True | REPO_DIR: /content/repo


In [2]:
# Veri kümesi: Drive'daki zip'i Colab'ın kendi diskine kopyalayıp açıyoruz.
# (Drive'dan binlerce küçük ses dosyasını tek tek okumak çok yavaş olurdu.)
DATASET = "XACLE_Challenge_2027_development_dataset"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    LOCAL_DIR = Path("/content/data")
    DATA_ROOT = LOCAL_DIR / DATASET
    if not DATA_ROOT.exists():                      # bu oturumda zaten açıldıysa atla
        candidates = [Path("/content/drive/MyDrive/Colab Notebooks/XACLE"),
                      Path("/content/drive/MyDrive/XACLE")]
        zip_path = next((d / f"{DATASET}.zip" for d in candidates
                         if (d / f"{DATASET}.zip").exists()), None)
        if zip_path is None:
            raise FileNotFoundError(f"{DATASET}.zip şu klasörlerde bulunamadı: {candidates}")
        LOCAL_DIR.mkdir(parents=True, exist_ok=True)
        print("Kopyalanıyor:", zip_path)
        subprocess.run(["cp", str(zip_path), str(LOCAL_DIR)], check=True)
        print("Açılıyor...")
        subprocess.run(["unzip", "-q", "-o", str(LOCAL_DIR / zip_path.name),
                        "-x", "__MACOSX/*", "-d", str(LOCAL_DIR)], check=True)
        (LOCAL_DIR / zip_path.name).unlink()
else:
    DATA_ROOT = REPO_DIR / "datasets" / DATASET

print(DATA_ROOT, "| bulundu:", DATA_ROOT.exists())

Mounted at /content/drive
Kopyalanıyor: /content/drive/MyDrive/Colab Notebooks/XACLE/XACLE_Challenge_2027_development_dataset.zip
Açılıyor...
/content/data/XACLE_Challenge_2027_development_dataset | bulundu: True


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from src.data import (load_metadata, load_all, audio_path, audio_info,
                      FILE_COL, TEXT_COL, MODEL_COL, LISTENER_COL, SCORE_COL, SPLITS)
from src.utils import compute_metrics
from src.models import MeanBaseline

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)

## 2. Veri kümesine genel bakış
Her örnek (instance) bir **(ses, metin)** çiftidir. Öznitelikler: ses dosyası (sayısal sinyal), metin açıklaması (string), üretici model (kategorik). Hedef değişken: ortalama uyum puanı (sayısal).

In [4]:
df = load_all(DATA_ROOT, average=True)          # ses başına ortalama puan
df_l = load_all(DATA_ROOT, average=False)       # dinleyici bazlı puanlar

print(df.groupby("split").size())
df.head()

split
test          3000
train         7500
validation    3000
dtype: int64


,wav_file_name,text,score,model_id,model_name,split
0,00000.wav,A water vehicle travels through the water with wind noise and a man talking,8.00,0,audiocaps,train
1,00001.wav,A motorcycle drives by,7.50,3,audioldm2,train
2,00002.wav,Some liquid flows followed by something sink,7.50,6,tango,train
3,00003.wav,Rain falling with distant thunder roaring,9.25,1,audiogen,train
4,00004.wav,The propellers of a helicopter scream as someone yells,2.00,6,tango,train


In [5]:
df.info()
print("\nEksik değerler:\n", df.isna().sum())
print("\nTekrar eden satır:", df.duplicated().sum())
print("Tekrar eden dosya adı:", df[FILE_COL].duplicated().sum())
print("Tekrar eden metin:", df[TEXT_COL].duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13500 entries, 0 to 13499
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   wav_file_name  13500 non-null  object 
 1   text           13500 non-null  object 
 2   score          13500 non-null  float64
 3   model_id       13500 non-null  int64  
 4   model_name     13500 non-null  object 
 5   split          13500 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 632.9+ KB

Eksik değerler:
 wav_file_name    0
text             0
score            0
model_id         0
model_name       0
split            0
dtype: int64

Tekrar eden satır: 0
Tekrar eden dosya adı: 0
Tekrar eden metin: 9064


**Bulgular — veriye genel bakış**

- Veri kümesinde **13.500 örnek** var: train 7.500, validation 3.000, test 3.000. Her örnek bir (ses, metin) çifti.
- **Öznitelik tipleri:**
  - `wav_file_name`: ses dosyasına referans
  - `text`: serbest metin (string)
  - `model_id` / `model_name`: **kategorik**. `model_id` sayı olarak kodlanmış ama sayısal bir büyüklük değil, model adını gösteriyor. Bu yüzden modellere sayısal girdi olarak verilmemeli.
  - `score`: 0–10 arası sürekli, sayısal **hedef değişken** (dinleyici puanlarının ortalaması)
- **Eksik değer yok**, tekrar eden satır ya da dosya adı da yok. Veri temizliği gerektirmiyor.
- **Tekrar eden metin sayısı 9.064**, yani 13.500 örnekte yalnızca 13.500 − 9.064 = **4.436 farklı açıklama** var. Her açıklama ortalama ~3 farklı ses üretim modeliyle seslendirilmiş. Bunun iki sonucu var:
  - Aynı açıklama için farklı modellerin sesleri doğrudan karşılaştırılabilir.
  - Modeli değerlendirirken aynı açıklamanın hem eğitimde hem testte olmamasına dikkat etmek gerekir. Metni ezberleyen bir model yanıltıcı derecede iyi görünebilir.